# Subfigure Extraction

This notebook extracts TEM subfigures from compound figures using a two-stage pipeline:

1. **YOLO** — detects and crops individual subfigures from each compound figure
2. **ResNet** — classifies each crop into one of four TEM modalities: `CTEM`, `HR-TEM`, `STEM`, `Diffraction`

Only crops classified as one of the target modalities are saved.

> **Note:** YOLO and ResNet model weights are not released in this repository as they are specific to our preprocessing pipeline. The scripts are provided for transparency. If you wish to retrain, refer to the settings below for model architecture and training configuration.

# Import packages

In [1]:
import os
import csv
import torch
from PIL import Image
from ultralytics import YOLO
from torchvision import transforms, models
from pathlib import Path

## Settings

Adjust the paths below to match your local setup:

- `FIGURE_DIR`: directory containing original compound figures downloaded from articles
- `SAVE_DIR`: directory where extracted TEM subfigures will be saved
- `CSV_PATH`: output CSV recording each subfigure and its predicted modality
- `YOLO_MODEL_PATH`: path to YOLO model weights
- `RESNET_MODEL_PATH`: path to ResNet classifier weights

ResNet configuration:
- Architecture: `resnet50`
- Input size: `384 x 384`
- Target classes: `CTEM`, `Diffraction`, `HR-TEM`, `STEM`

In [ ]:
FIGURE_DIR        = "/path/to/your/raw_figures"
SAVE_DIR          = "/path/to/your/TEM_figures"
CSV_PATH          = "/path/to/your/results.csv"

YOLO_MODEL_PATH   = "/path/to/your/yolo_extractor.pt"
RESNET_MODEL_PATH = "/path/to/your/resnet_classifier.pth"

MODEL_ARCH        = "resnet50"
IMAGE_SIZE        = 384
CONF_THRESHOLD    = 0.0
TARGET_CLASSES    = ['CTEM', 'Diffraction', 'HR-TEM', 'STEM']
SAVE_CROPS        = True

## Load Models

Loads YOLO for subfigure detection and ResNet for modality classification.
Automatically uses GPU if available.

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CSV_PATH) or ".", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

print("Loading YOLO model...")
yolo = YOLO(YOLO_MODEL_PATH)

print("Loading ResNet model...")
ckpt = torch.load(RESNET_MODEL_PATH, map_location=device)

class_names = ckpt["class_names"]
num_classes = len(class_names)

model = models.resnet50()
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

print("ResNet classes:", class_names)

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Extract and Classify Subfigures

For each compound figure in `FIGURE_DIR`:
1. YOLO detects bounding boxes of individual subfigures
2. Each crop is classified by ResNet
3. Crops matching `TARGET_CLASSES` are saved to `SAVE_DIR` and recorded in `CSV_PATH`

In [ ]:
supported_ext = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp")

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["source_image", "sub_image", "resnet_pred_class"])
    f.flush()
    
    idx = 0
    for src_name in sorted(os.listdir(FIGURE_DIR)):
        if not src_name.lower().endswith(supported_ext):
            continue

        src_path = os.path.join(FIGURE_DIR, src_name)
        idx+=1
        print(f"\nProcessing {idx} image: {src_name}")

        img = Image.open(src_path).convert("RGB")
        results = yolo(src_path, verbose=False)
        boxes = results[0].boxes

        if boxes is None or len(boxes) == 0:
            print("  No detections.")
            continue

        base = os.path.splitext(src_name)[0]
        W, H = img.size

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
            # Clamp to image boundaries
            x1 = max(0, min(x1, W - 1))
            y1 = max(0, min(y1, H - 1))
            x2 = max(0, min(x2, W))
            y2 = max(0, min(y2, H))
        
            # bbox check
            if x2 <= x1 or y2 <= y1:
                print(f"  skip invalid bbox: {(x1, y1, x2, y2)}")
                continue
        
            crop = img.crop((x1, y1, x2, y2)).convert("RGB")
        
            # ResNet classification
            x = transform(crop).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model(x)
                pred_idx = int(torch.argmax(logits, dim=1).item())
        
            pred_class = class_names[pred_idx]
        
            sub_name = f"{base}_crop{i}.png"
            sub_path = os.path.join(SAVE_DIR, sub_name)
        
            if pred_class in TARGET_CLASSES:
                crop.save(sub_path, format="PNG")
                print(f"  crop {i}: {pred_class} -> SAVED {sub_name}")
                writer.writerow([src_name, sub_name, pred_class])
                f.flush()
            else:
                print(f"  crop {i}: {pred_class} -> skipped")
        
            del x, logits

print(f"\nDone. CSV saved to: {CSV_PATH}")